In [ ]:
# Automatically reloads all code changes in python modules when changes are saved
%load_ext autoreload
%autoreload 2

In [ ]:
## Initialize logger for whole project
## Also tracks output of YOLO ML training to this log file

from flower_classifier.logging import setup_logging
from ultralytics import YOLO

# LOG_FILE = 'flower_classifier.log'
# logger = setup_logging(log_file=str(LOG_FILE))
logger = setup_logging()
logger.debug(f"Logging setup successful.")

In [ ]:
## Sets up environment variables with kaggle
## MUST HAVE .env FILE WITH FILLED IN KAGGLE LOGIN INFO EVEN IF USE_KAGGLE IS FALSE
from dotenv import load_dotenv
from pathlib import Path
from flower_classifier.config_IO import PROJECT_ROOT
import os

file_path = PROJECT_ROOT / ".env"
if not os.path.isfile(file_path):
    msg = "'.env' file not found in the project root. Please make .env file with KAGGLE_USERNAME and KAGGLE_KEY. See README.md for more information."
    logger.error(msg)
    raise FileNotFoundError(msg)
else:
    logger.info(".env file found! Importing configs.")

# Extract environment variables from .env file
load_dotenv()
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')
GOOGLE_ZIP_ID = os.getenv('GOOGLE_ZIP_ID')

# # Resolve project root directory
# path = Path(__name__).resolve().parent
# while path.name != "flower-classifier" and path != path.parent:
#     path = path.parent
# logger.info(f"Path to project parent directory: '{path}'")
# PROJECT_ROOT = path

In [ ]:
from flower_classifier.data_setup import download_dataset, unzip_dataset, make_val_and_test_sets
from flower_classifier.config_IO import format_config_string, save_json_config, load_json_config, make_YOLO_config, get_highest_numbered_run
from flower_classifier.config_IO import print_system_info, verify_venv
from flower_classifier.file_io import view_image, view_text

# import get_dataset
# import ingest
# import config_IO
# import system_info
# import view_file

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

In [ ]:
try:
    verify_venv()
except Exception as e:
    logger.exception(f"Exception while verifying venv: \n{e}")
    raise e
    
logger.info("Venv verified successfully")

In [ ]:
USE_KAGGLE = False

zip_location = PROJECT_ROOT / 'data' / 'compressed' / 'tomato-flower-3-class.zip'

if not zip_location.is_file():
    logger.info(f"Zip file not previously found, beginning retrieval.")
    
    do_unzip = download_dataset(from_kaggle=USE_KAGGLE, kaggle_username=KAGGLE_USERNAME, kaggle_key=KAGGLE_KEY, google_zip_id=GOOGLE_ZIP_ID)
    
    unzip_dataset()
    
else:
    logger.info(f"zip file already exists at '{zip_location}', skipping download.")

In [ ]:
# If image set 'valid' has not yet been split into 'val' and 'test' has not already been done, do that now.

# The tomato flower image dataset is sourced from https://www.kaggle.com/datasets/jawadulkarim117/tomato-flower-3-class along with info about the accompanying paper.
# That experiment used a pure 80/20 split between training and validation, with no testing set.
# This experiment requires an test set, so we split the 'valid' dataset into two new equally-sized 'test' and 'val' sets.

# New split: 80/10/10 for training/validation/test.

if not os.path.exists(str(PROJECT_ROOT / 'data/data_set/tomato-flower/val/images')) and not os.path.exists(str(PROJECT_ROOT / 'data/data_set/tomato-flower/test/images')):
    logger.info("Image set 'valid' set not yet split into distinct 'val' and 'test' sets. Beginning split now.")
    make_val_and_test_sets()
else:
    logger.info("Split into 'val' and 'test' done previously. Skipping split.")

logger.info("Dataset setup and ingestion complete.")

In [ ]:
print_system_info()

In [ ]:
## Creates the BASIC YOLO configs, with only a few values changed by default.
## Any other parameter changes happen in the overrides below, or for setting the actual model directly before training.
config_location =  str(PROJECT_ROOT / "config/YOLO_config.json")
make_YOLO_config(file_path=config_location)

In [ ]:
## Load parameters (as dict) from specified file location
## You can change the path if you want to use different basic configs than created in the previous cell, or set them directly in the cell below.

TRAIN_CONFIG = load_json_config(config_location)

In [ ]:
## Manual config overrides

# The following are already modified from default inside of make_YOLO_config:
# "batch": -1,           (-1 does autoselect, changed from default 16)
# "device": -1,          (-1 does autoselect idle GPUs, changed from default <blank>)
# "pretrained": False,   (restart training from scratch. Changed from True)


## Some of your manual changes might be, for example:
# TRAIN_CONFIG["train_params"]["epochs"] = 150
# TRAIN_CONFIG["train_params"]["lr0"] = 0.005

## Follow a cosine-based simulated annealing (decay) of learning rates across epochs from lr0 to lrf:
# TRAIN_CONFIG["train_params"]["lr0"] = 0.03
# TRAIN_CONFIG["train_params"]["lrf"] = 0.001
# TRAIN_CONFIG["train_params"]["cos_lr"] = True


## CHANGES TO DEFAULTS
TRAIN_CONFIG["train_params"]["epochs"] = 100
TRAIN_CONFIG["train_params"]["patience"] = 3

## RUN SEED
# Seed will be set in the run_yolo_training function call. Initially set here to note it will be used.
# Using "deterministic" for reproducibility.
TRAIN_CONFIG["train_params"]["seed"] = 0
TRAIN_CONFIG["train_params"]["deterministic"] = True

## We change the configs within the 'TRAIN_CONFIG' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file for future reference.
logger.info(f"Full config set after all manual overrides: \n{format_config_string(config=TRAIN_CONFIG)}")

In [ ]:
## CPU/GPU Usage Config

CPU_COUNT = multiprocessing.cpu_count()
DATALOADER_WORKERS = TRAIN_CONFIG["train_params"].get("workers", 8)
CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, DATALOADER_WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

print(f"CPU count: {CPU_COUNT}")
print(f"DataLoader workers: {DATALOADER_WORKERS}")
print(f"CPU threads per process: {CPU_THREADS_PER_PROC}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
## PREFLIGHT CHECKS
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.


# ==========================
# DIRECTORY CONFIGS
# ==========================
DATA_YAML = PROJECT_ROOT / "config/data.yaml"
DATASET_ROOT = PROJECT_ROOT / "data/data_set/tomato-flower"
PROJECT_DIR = PROJECT_ROOT / "runs"

TRAIN_PARAMS: dict = TRAIN_CONFIG["train_params"]

# ==========================
# CPU THREAD CONTROL (GPU TRAINING SAFE)
# ==========================
CPU_COUNT = multiprocessing.cpu_count()
WORKERS = TRAIN_PARAMS.get("workers", 8)

CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

logger.info(
    f"\n=== CPU THREAD CONFIG ==="
    f"\n CPU count              : {CPU_COUNT}"
    f"\n DataLoader workers     : {WORKERS}"
    f"\n Threads per process    : {CPU_THREADS_PER_PROC}"
    f"\n torch num threads      : {torch.get_num_threads()}"
    f"\n"
)

# ==========================
# SYSTEM INFO
# ==========================
logger.info(
    f"\n=== SYSTEM INFO ==="
    f"\n Python executable : {sys.executable}"
    f"\n Python version    : {sys.version.splitlines()[0]}"
    f"\n Platform          : {platform.platform()}"
    f"\n Machine           : {platform.machine()}"
    f"\n Processor         : {platform.processor()}"
    f"\n CPU count         : {CPU_COUNT}"
    f"\n Working dir       : {Path.cwd()}"
    f"\n"
)

# ==========================
# TRAIN CONFIG CHANGES
# ==========================
logger.info(
    f"\n=== TRAIN CONFIG CHANGES ==="
    f"\n {TRAIN_CONFIG}"
    #f"\n model_cfg   : {TRAIN_CONFIG['model_cfg']}"
    #f"\n data_yaml   : {TRAIN_CONFIG['data_yaml']}"
    #f"\n project_dir : {TRAIN_CONFIG['project_dir']}"
    #f"\n run_name    : {TRAIN_CONFIG['run_name']}"
    #f"\n epochs      : {TRAIN_PARAMS['epochs']}"
    #f"\n imgsz       : {TRAIN_PARAMS['imgsz']}"
    #f"\n batch       : {TRAIN_PARAMS['batch']}"
    #f"\n device      : {TRAIN_PARAMS['device']}"
    #f"\n workers     : {TRAIN_PARAMS['workers']}"
    #f"\n amp         : {TRAIN_PARAMS['amp']}"
    #f"\n optimizer   : {TRAIN_PARAMS['optimizer']}"
    #f"\n pretrained  : {TRAIN_PARAMS['pretrained']}"
    f"\n"
)

# ==========================
# TORCH / CUDA INFO
# ==========================
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

logger.info(
    f"\n=== TORCH INFO ==="
    f"\n torch version       : {torch.__version__}"
    f"\n CUDA available      : {cuda_available}"
    f"\n CUDA device count   : {torch.cuda.device_count() if cuda_available else 0}"
    f"\n MPS available       : {mps_available}"
    f"\n"
)

if cuda_available:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_mem_gb = props.total_memory / (1024 ** 3)

        logger.info(
            f"\nCUDA DEVICE {i}:"
            f"\n name              = {torch.cuda.get_device_name(i)}"
            f"\n capability        = {props.major}.{props.minor}"
            f"\n total_memory_gb   = {total_mem_gb:.2f}"
            f"\n multiprocessors   = {props.multi_processor_count}"
        )
else:
    logger.warning("CUDA is NOT available. Training will fall back to CPU.")

# ==========================
# CUDA SMOKE TEST
# ==========================
logger.info(f"Testing CUDA usage...")
if cuda_available:
    try:
        x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
        logger.info(f"CUDA smoke test PASSED (tensor device = {x.device})")
        del x
        torch.cuda.empty_cache()
    except Exception as e:
        logger.exception(f"CUDA detected but failed runtime test: {e}")
        raise RuntimeError(f"CUDA detected but failed runtime test: {e}")

# ==========================
# ENV VARS
# ==========================
cuda_visible_devices = os.environ.get('CUDA_VISIBLE_DEVICES')
logger.info(
    f"\n=== IMPORTANT ENV VARS ==="
    f"\n OMP_NUM_THREADS        = {os.environ.get('OMP_NUM_THREADS')}"
    f"\n MKL_NUM_THREADS        = {os.environ.get('MKL_NUM_THREADS')}"
    f"\n NUMEXPR_NUM_THREADS    = {os.environ.get('NUMEXPR_NUM_THREADS')}"
    f"\n OPENBLAS_NUM_THREADS   = {os.environ.get('OPENBLAS_NUM_THREADS')}"
    f"\n CUDA_VISIBLE_DEVICES   = {cuda_visible_devices if cuda_visible_devices is not None else '<Not restricted>'}"
    f"\n ULTRALYTICS_CONFIG_DIR = {os.environ.get('ULTRALYTICS_CONFIG_DIR')}"
    f"\n"
)

if os.environ.get("CUDA_VISIBLE_DEVICES") is not None:
    logger.info(
        f"CUDA_VISIBLE_DEVICES is set to {os.environ.get('CUDA_VISIBLE_DEVICES')} "
        "→ This may restrict visible GPUs."
    )

# ==========================
# DEVICE VALIDATION
# ==========================
requested_device = TRAIN_PARAMS["device"]

if requested_device != "cpu" and not cuda_available:
    raise RuntimeError(
        f"Config requests GPU (device={requested_device}) but CUDA is unavailable."
    )

if requested_device == -1:
    logger.info("device=-1 → auto-selecting most idle GPU.")

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    logger.info(
        f"\n{label}:"
        f"\n path    = {path}"
        f"\n exists  = {path.exists()}"
        f"\n is_dir  = {path.is_dir()}"
        f"\n is_file = {path.is_file()}"
    )

# ==========================
# PATH CHECKS
# ==========================
logger.info("\n=== PATH CHECKS ===")

print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)

    img_count = count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    lbl_count = count_files(lbl_dir, {".txt"})

    logger.info(
        f"{split.upper()} COUNTS:"
        f"\n image files = {img_count}"
        f"\n label files = {lbl_count}"
    )

# ==========================
# YAML VALIDATION
# ==========================
logger.info("\n=== YAML CONTENTS ===")

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

logger.info(json.dumps(data_cfg, indent=2))

logger.info("\n=== YAML PATH VALIDATION ===")

for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    logger.info(
        f"{key}: {p}"
        f"\n exists = {p.exists()}"
        f"\n is_dir = {p.is_dir()}"
    )

logger.info("\n=== PREFLIGHT CHECK COMPLETE ===\n")

In [ ]:
def verify_models_accessible(models: list[str]) -> bool:
    """
    Verify all specified YOLO models have a valid name format and are accessible.

    Returns True if valid and accessible, otherwise returns False.
    """

    all_passed: bool = True 
    
    for model in models:
        try:
            verify_model = YOLO(f"{model}.yaml")
            logger.info(f"YOLO model config and file '{model}' recognized. (Has valid name, version, and scale).")
            del verify_model
        except Exception as e:
            logger.exception(f"ERROR! Model config '{model}' NOT recognized! \n{e}")
            all_passed = False

    return all_passed
    

In [ ]:
def run_yolo_training(training_config: dict) -> bool:    
    
    ## Get current run number from the file names
    using_model: str = training_config['model_cfg'].split(".yaml")[0].split(".pt")[0]
    RUN_NUM: int = get_highest_numbered_run(
        parent_path=training_config["project_dir"],
        model_name=using_model
    )
    logger.info(f"Highest previous run of {using_model} found is {RUN_NUM}.")

    
    
    ## Training the YOLO26 model
    ## Run number is automatically incremented based on the folders that exists in the output directory.
    ## This is due to 'exist_ok=False' instead of True, where it would just replace the dir instead of making an incremented new one. 
    RUN_NUM += 1
    
    import gc
    model = None
    results = None
    try:
        model = YOLO(training_config["model_cfg"])
        run_name = str(training_config['run_name']) + str(RUN_NUM)
        
        logger.info(f"Beginning training run '{run_name}'")
        
        # Set seed based on the current run number for this model. Deterministic (for reproducibility) as set above.
        training_config["train_params"]["seed"] = RUN_NUM
        logger.info(f"Using seed {training_config['train_params']['seed']} with deterministic order for this training.")
    
        results = model.train(
            data=training_config["data_yaml"],
            project=training_config["project_dir"],
            name=run_name,
            **training_config["train_params"]
        )
    
        logger.info(f"DONE! Model training run {str(training_config['run_name']) + str(RUN_NUM)} complete!")
        
        
        ## Save results and configs to corresponding `run` subdirectory
        import shutil
        CUR_RUN_ROOT = PROJECT_ROOT / f"runs/exp_{using_model}_{RUN_NUM}/"
        
        ## Save current config values back to specified config json file.
        config_path = CUR_RUN_ROOT / "configs_changed_from_default.json"
        
        logger.info(f"Saving CHANGED configs from default for run 'exp_{using_model}_{RUN_NUM}' to location '{str(config_path)}' now...")
        save_json_config(training_config, config_path)
        logger.info("Configs saved successfully!")
        
        ## Copy current log file
        logger.info("Copying current .log file into finished training location...")
        
        global_log = PROJECT_ROOT / "logs/flower_classifier.log"
        cur_run_log = str(CUR_RUN_ROOT / "flower_classifier.log")
        
        shutil.copy2(global_log, cur_run_log)
        
        logger.info(f"Log copied to '{cur_run_log}' successfully!")

    
    except Exception as e:
            logger.exception(f"ERROR while training or saving data from model run '{str(training_config['run_name']) + str(RUN_NUM)}': \n{e}")
            return False
        
    finally:
        ## Clear the previous results and model from memory using Python's Garbage Collector.
        ## Since this training is intended to be run multiple times, back-to-back, this helps manage memory usage and ensure all trainings can proceed.
        logger.info(f"Clearing runtime memory from previous model training...")
        del results
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info(f"Training memory cleared for next run!")


    logger.info(f"Training run {str(training_config['run_name']) + str(RUN_NUM)} completed successfully!")
    return True

In [ ]:
## SET YOUR TARGET YOLO MODEL(S) HERE
## Some possible values: "yolo26n", "yolo26m", "yolov8x", "yolov9e", etc.
## If you want to test the same model multiple times, include it in the list that many times!
#using_models: list[str] = ["yolo26n"]*3 + ["yolo26m"]*3 + ["yolo26x"]*3   +   ["yolov8n"]*3 + ["yolov8m"]*3 + ["yolov8x"]*3
using_models: list[str] = ["yolo11n"]*3 + ["yolo11m"]*3 + ["yolo11x"]*3   +   ["yolov5nu"]*3 + ["yolov5mu"]*3 + ["yolov5xu"]*3

## A full list of available models (and details thereof) is available at the Ultralytics docs page, linked here:
## https://docs.ultralytics.com/models


## Ensure all specified models are valid before running ANY trainings (fail early).
if not verify_models_accessible(models=using_models):
    logger.warning(f"Training canceled since specified one or more specified models were inaccessible!")
    raise


## Iterate through all models in the provided list
for model in using_models:
    
    ## Update configs for the selected model
    TRAIN_CONFIG["model_cfg"] = f"{model}.yaml"
    TRAIN_CONFIG["run_name"] = f"exp_{model}_"
    
    
    ## Run training on selected model
    was_success = run_yolo_training(training_config=TRAIN_CONFIG)

    if not was_success:
        logger.info(f"Canceling training loop since a model run failed.")
        raise

In [ ]:
# Test only best.pt from immediate exp_* folders, sequentially.
# This cell can run independently; no training/setup cells need to be rerun.
from pathlib import Path
import json
from flower_classifier.config_IO import PROJECT_ROOT
from flower_classifier.evaluation import evaluate_best_checkpoints


def run_yolo_testing(config_path=None, *, batch=16, device=None, imgsz=640, workers=8):
    """Evaluate all exp_*/weights/best.pt on the configured test split.

    Saves plots, per-model metrics.json, and a combined summary.csv beneath
    test_results/<timestamp>/. Each call creates a new results directory.
    device=None lets Ultralytics select the device on this computer.
    """
    config_path = Path(config_path) if config_path else PROJECT_ROOT / "config/YOLO_config.json"
    with config_path.open() as handle:
        config = json.load(handle)
    return evaluate_best_checkpoints(
        runs_dir=config["project_dir"],
        data_yaml=config["data_yaml"],
        batch=batch, device=device, imgsz=imgsz, workers=workers,
    )


# Run this line when ready to evaluate all trained models under `runs/exp_*` subdirectories:
#test_results = run_yolo_testing()

In [ ]:
# Afterwards (or in a fresh session), compare saved trials without rerunning YOLO:
from flower_classifier.evaluation import compare_model_results, get_latest_summary_csv

# Leave as None to use the newest timestamped evaluation summary.
# To override, set an explicit path to an earlier summary.csv.
path_to_summary_csv: Path | None = None
# path_to_summary_csv = PROJECT_ROOT / "test_results/<timestamp>/summary.csv"

if path_to_summary_csv is None:
    path_to_summary_csv = get_latest_summary_csv(PROJECT_ROOT / "test_results")
print(f"Using evaluation summary: {path_to_summary_csv}")

# Use the summary.csv path printed by run_yolo_testing():
model_stats, comparison_figure = compare_model_results(
    summary_csv=path_to_summary_csv,

    # use one or the other:
    statistic="mean",
    #statistic="median",
    
    # use one or the other:
    metric="metrics/mAP50(B)"
    #metric="metrics/mAP50-95(B)",
)
display(model_stats)
display(comparison_figure)